# The endogenous gridpoint method

Notes at <https://dse.iskh.me/egm>.

## EGM for the consumption-savings model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def lin(x, xp, fp):
    '''Linear interpolation, extrapolated linearly beyond the last point'''
    slope = (fp[-1] - fp[-2]) / (xp[-1] - xp[-2])
    return np.where(x > xp[-1], fp[-1] + slope * (x - xp[-1]), np.interp(x, xp, fp))

def egm_iterations(beta, R, y, K, A=np.linspace(0, 10, 5), origin=False, title=''):
    '''Plot the first K iterations of EGM with log utility and deterministic income y'''
    M0, c0 = np.array([0., 10.]), np.array([0., 10.])     # initial policy: consume everything
    fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
    ax[2].plot(M0, c0, 'o-', ms=4, lw=0.8, label='iteration 0')
    for i in range(1, K + 1):
        c1 = np.maximum(lin(R * A + y, M0, c0), 1e-10)    # next period consumption, kept positive
        c = c1 / (beta * R)                               # inverted Euler equation: 1/c = beta R / c1
        M = A + c                                         # endogenous grid points
        if origin:                                        # add the origin: the constrained segment
            M, c, Ap = np.r_[0, M], np.r_[0, c], np.r_[np.nan, A]
        else:
            Ap = A
        for k, (x, yy) in enumerate(((Ap, c), (Ap, M), (M, c))):
            ax[k].plot(x, yy, 'o-', ms=4, lw=0.8, label=f'iteration {i}')
        M0, c0 = M, c
    for a, (xl, t) in zip(ax, (('savings A', 'consumption c(A)'), ('savings A', 'wealth M(A)'),
                                ('wealth M', 'consumption c(M)'))):
        a.set(xlabel=xl, title=t); a.grid(color='0.9')
    ax[2].legend(fontsize=8)
    fig.suptitle(title)
    plt.show()

egm_iterations(beta=0.95, R=1.0, y=0.0, K=4, title='Cake eating, beta = 0.95, R = 1, y = 0')

## Corner solutions

In [ ]:
egm_iterations(beta=0.95, R=1.05, y=1.0, K=12, origin=True,
               title='Consumption-savings with income, beta = 0.95, R = 1.05, y = 1, origin added')

## Implementation and comparison

In [ ]:
import sys, time, pathlib, tempfile, urllib.request
CODE = pathlib.Path('../session16-oct20')          # the code of Class 16
if not CODE.exists():                                     # otherwise fetch it from the code repository
    CODE = pathlib.Path(tempfile.mkdtemp())
    urllib.request.urlretrieve('https://raw.githubusercontent.com/fediskhakov/sb-dse-code/main/session16-oct20/deaton.py',
                               CODE / 'deaton.py')
sys.path.insert(0, str(CODE))
from deaton import deaton, interp, solve_vfi, bellman_discretized, bellman_continuous, \
                   solve_timeiter, accuracy, simulate

In [ ]:
def solve_egm(m, maxiter=500, tol=1e-4, callback=None):
    '''EGM for the consumption-savings model, returns the policy on the state grid m.grid'''
    A = np.linspace(0, m.Mbar, m.ngrid)                   # grid on savings, the post-decision state
    M0, c0 = np.array([0., m.Mbar]), np.array([0., m.Mbar])   # initial policy: consume everything
    for i in range(maxiter):
        M1 = m.next_wealth(A[:, None], 0, m.quadp[None, :])   # next period wealth: savings x income nodes
        c1 = np.maximum(interp(M1, M0, c0), m.eps)        # next period consumption, kept positive
        rhs = m.beta * m.R * (m.mu(c1) @ m.quadw)         # RHS of the Euler equation
        c = np.r_[0., m.imu(rhs)]                         # inverted Euler equation, origin added
        M = np.r_[0., A + c[1:]]                          # endogenous grid points
        err = np.max(np.abs(interp(m.grid, M, c) - interp(m.grid, M0, c0)))
        if callback: callback(iter=i, M=M, c=c, err=err)
        if err < tol: break
        M0, c0 = M, c
    else:
        raise RuntimeError('No convergence: maximum number of iterations achieved')
    return interp(m.grid, M, c)                           # policy on the fixed grid, to compare

In [ ]:
print(f'{"grid":>5} {"solver":<25} {"time, s":>9} {"Euler residual":>15}')
for n in (50, 100, 200):
    m = deaton(ngrid=n)
    for name, solver in (('VFI, discretized choice', lambda: solve_vfi(m, bellman_discretized)),
                         ('VFI, continuous choice', lambda: solve_vfi(m, bellman_continuous)),
                         ('time iterations', lambda: solve_timeiter(m)),
                         ('EGM', lambda: solve_egm(m))):
        t0 = time.perf_counter()
        c = solver()
        print(f'{n:>5} {name:<25} {time.perf_counter() - t0:>9.4f} {accuracy(m, c):>15.2e}')

k = deaton(beta=0.9, R=1/0.9, sigma=1e-10, nquad=2)       # Keane's test: deterministic, beta*R = 1
_, C = simulate(k, solve_egm(k, tol=1e-10, maxiter=5000), 5.0, 100)
print(f"\nKeane's test for EGM: max |c_t / c* - 1| = {np.max(np.abs(C[0] / (((k.R - 1)*5 + 1)/k.R) - 1)):.1e}")